# MNIST CPU Benchmark
Measures FPS, accuracy and power on the KV260 **ARM CPU (Cortex-A53)**.

**Task**: Classify handwritten digits (0-9) from 28x28 grayscale images.

**Run from any terminal on the KV260** (SSH or Jupyter terminal)

Prerequisites:
- Run `bash mnist/setup.sh` first to download MNIST dataset
- onnxruntime installed

Expected: **~2060 FPS, 98% accuracy, 5.57W, 370 FPS/Watt**

> Note: MNIST is tiny — CPU is fast enough that DPU advantage is only 1.4x.
> DPU shines on larger models: ResNet50 (29x), YOLOv3 (50x), InceptionV1 (30x).

In [ ]:
import os, time, gzip, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
DATA_DIR     = "/home/ubuntu/mnist_data"
N_WARMUP     = 5
N_BENCHMARK  = 100

# Smart path: check local models/ first, then /home/ubuntu/
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("cpu_bench.ipynb"))
MODEL_LOCAL  = os.path.join(NOTEBOOK_DIR, "models", "mnist-12.onnx")
MODEL_UBUNTU = "/home/ubuntu/mnist-12.onnx"
MODEL_PATH   = MODEL_LOCAL if os.path.exists(MODEL_LOCAL) else MODEL_UBUNTU

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Using model: {MODEL_PATH}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")
print(f"MNIST data exists: {os.path.exists(DATA_DIR)}")
print(f"Current power: {read_power_mw()/1000:.2f} W (idle)")

## Step 1 — Load MNIST Test Data
> Note: CPU ONNX model expects **(1, 1, 28, 28)** — channel first. DPU uses (1, 28, 28, 1) — channel last.

In [ ]:
def load_images(path):
    with gzip.open(path, 'rb') as f:
        f.read(16)
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(-1, 28, 28)

def load_labels(path):
    with gzip.open(path, 'rb') as f:
        f.read(8)
        return np.frombuffer(f.read(), dtype=np.uint8)

images = load_images(f"{DATA_DIR}/t10k-images-idx3-ubyte.gz")
labels = load_labels(f"{DATA_DIR}/t10k-labels-idx1-ubyte.gz")

# CPU ONNX expects (1, 1, 28, 28) float32 — channel first
data = images.astype(np.float32)[:, np.newaxis, :, :] / 255.0

print(f"Loaded: {images.shape[0]} images")
print(f"Data shape: {data.shape}  (N, C, H, W) — channel first for ONNX")

## Step 2 — Load MNIST ONNX Model

In [ ]:
import onnxruntime as ort
print(f"ONNX Runtime version: {ort.__version__}")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}. Run setup.sh first.")

session = ort.InferenceSession(MODEL_PATH, providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
print(f"Input: {input_name} {session.get_inputs()[0].shape}")
print(f"Output: {session.get_outputs()[0].shape}  (10 digit scores)")

## Step 3 — Warmup

In [ ]:
for i in range(N_WARMUP):
    session.run(None, {input_name: data[i:i+1]})
print("Warmup done!")

## Step 4 — Benchmark with Accuracy + Power

In [ ]:
correct = 0
power_samples = []
start = time.time()
for i in range(N_BENCHMARK):
    result = session.run(None, {input_name: data[i:i+1]})
    pred = int(np.argmax(result[0]))
    if pred == int(labels[i]):
        correct += 1
    if i % 10 == 0:
        power_samples.append(read_power_mw())
elapsed = time.time() - start
print("Done!")

## Step 5 — Results

In [ ]:
avg_power_w  = sum(power_samples) / len(power_samples) / 1000
fps          = N_BENCHMARK / elapsed
latency_ms   = elapsed / N_BENCHMARK * 1000
fps_per_watt = fps / avg_power_w
accuracy     = correct / N_BENCHMARK * 100

print("=" * 40)
print("MNIST CPU BENCHMARK RESULTS")
print("=" * 40)
print(f"Platform:    KV260 ARM CPU (Cortex-A53)")
print(f"Accuracy:    {accuracy:.1f}%  ({correct}/{N_BENCHMARK} correct)")
print(f"FPS:         {fps:.1f}")
print(f"Latency:     {latency_ms:.2f} ms/frame")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)
print(f"Expected:    ~2060 FPS, 98% accuracy, 5.57W, ~370 FPS/Watt")
print(f"DPU result:  ~2863 FPS, 99% accuracy, 5.61W, ~510 FPS/Watt  (1.4x more efficient)")